In [1]:
!pip install -q langchain-groq langchain-core langchain-community pypdf

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "Resume_Screener_Assignment"

In [3]:
!mkdir -p prompts chains

In [4]:
%%writefile prompts/templates.py
from langchain_core.prompts import ChatPromptTemplate

SCREENER_SYSTEM_PROMPT = """
You are a Technical Recruiter expert. Evaluate the Resume text against the Job Description (JD).

TASK:
1. Extract: Technical skills, Tools, and Years of Experience.
2. Matching: Compare the Resume content strictly against JD requirements.
3. Scoring: Assign a score (0-100) based on relevance.
4. Explanation: Provide a brief justification for the score.

CONSTRAINTS:
- DO NOT assume skills. If a skill isn't in the text, it doesn't exist.
- Output MUST be valid JSON.

{format_instructions}
"""

user_template = "Job Description:\n{jd}\n\nResume Content:\n{resume}"

def get_screener_prompt(parser):
    return ChatPromptTemplate.from_messages([
        ("system", SCREENER_SYSTEM_PROMPT),
        ("user", user_template)
    ]).partial(format_instructions=parser.get_format_instructions())

Overwriting prompts/templates.py


In [5]:
%%writefile chains/screener.py
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser
# Fixed the import path here
try:
    from langchain_core.pydantic_v1 import BaseModel, Field
except ImportError:
    from pydantic import BaseModel, Field
from typing import List
from prompts.templates import get_screener_prompt

# Define JSON structure for the Bonus requirement
class ScreeningResult(BaseModel):
    extracted_skills: List[str] = Field(description="List of skills found in resume")
    experience_years: float = Field(description="Total years of relevant experience")
    fit_score: int = Field(description="Score from 0 to 100")
    explanation: str = Field(description="Reasoning for the score")

# Initialize Parser and LLM
parser = JsonOutputParser(pydantic_object=ScreeningResult)
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

# Build LCEL Chain
screener_chain = get_screener_prompt(parser) | llm | parser

Overwriting chains/screener.py


In [6]:
from google.colab import files
from pypdf import PdfReader
import io
import json
from chains.screener import screener_chain

def extract_pdf_text():
    uploaded = files.upload()
    if not uploaded: return None
    name = list(uploaded.keys())[0]
    reader = PdfReader(io.BytesIO(uploaded[name]))
    return "".join([page.extract_text() for page in reader.pages])

jd_text = """
Role: Junior AI/ML Engineer
Technical Requirements: > - Strong proficiency in Python and FastAPI.

Hands-on experience with LangChain and Groq/OpenAI APIs.

Experience building RAG (Retrieval-Augmented Generation) pipelines.

Knowledge of Vector Databases (ChromaDB/Pinecone).

Education: Pursuing or completed B.E. in CS/AI/ML.
"""

print("🚀 Starting AI Resume Screening Pipeline...")


candidate_types = ["Strong", "Average", "Weak"]
all_results = []

for c_type in candidate_types:
    print(f"\n[ACTION] Upload the {c_type} candidate Resume PDF:")
    resume_text = extract_pdf_text()

    if resume_text:
        response = screener_chain.invoke(
            {"jd": jd_text, "resume": resume_text},
            config={"tags": [c_type, "Colab_Run"]}
        )
        all_results.append({"type": c_type, "data": response})
        print(f"✅ {c_type} Candidate Scored: {response['fit_score']}")
    else:
        print(f"❌ Skipped {c_type} candidate.")

print("\n" + "="*40)
print("FINAL SCREENING SUMMARY")
print("="*40)
for res in all_results:
    print(f"{res['type']}: {res['data']['fit_score']}/100 | {res['data']['explanation'][:100]}...")

🚀 Starting AI Resume Screening Pipeline...

[ACTION] Upload the Strong candidate Resume PDF:


Saving Resume 1.pdf to Resume 1.pdf
✅ Strong Candidate Scored: 80

[ACTION] Upload the Average candidate Resume PDF:


Saving Resume 2.pdf to Resume 2.pdf


✅ Average Candidate Scored: 20

[ACTION] Upload the Weak candidate Resume PDF:


Saving Resume 3.pdf to Resume 3.pdf
✅ Weak Candidate Scored: 0

FINAL SCREENING SUMMARY
Strong: 80/100 | The candidate's skills and experience align well with the job requirements, including proficiency in...
Average: 20/100 | The candidate has basic Python skills which is a requirement, but lacks experience in FastAPI, LangC...
Weak: 0/100 | The resume does not mention any technical skills or experience relevant to the Junior AI/ML Engineer...
